In [5]:
import torch

import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))




from utilities.model import CNNMLPModel


from python.dataset_creation import (unique_subjects, all_subjects, 
                                    X_glob,X_seq, y                                        
                                    )


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNMLPModel(
    input_dim_seq=15,
    input_dim_global=9,
    num_classes=3
).to(device)

fold_results = torch.load("fold_results.pth", map_location=device)

checkpoint = fold_results["s2"]
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()



C:\Users\aless\AppData\Local\Temp\ipykernel_6588\3736214503.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fold_results = torch.load("fold_results.pth", map_location=de

CNNMLPModel(
  (features): Sequential(
    (0): Conv1d(15, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Dropout(p=0.3, inplace=False)
    (11): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
    (12): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): AdaptiveAvgPool1d(output_size=1)
  )
  (global_mlp): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=Tr

In [7]:
val_subject = 's1'   # change this to the patient you want to leave out

train_indices = [
    i for i, s in enumerate(all_subjects)
    if s != val_subject
]

val_indices = [
    i for i, s in enumerate(all_subjects)
    if s == val_subject
]

X_seq_train = X_seq[train_indices]
X_seq_val = X_seq[val_indices]

X_glob_train = X_glob[train_indices]
X_glob_val = X_glob[val_indices]

y_train = y[train_indices]
y_val = y[val_indices]

In [8]:
model.eval()

with torch.no_grad():
    X_seq_val = X_seq_val.to(device)
    X_glob_val = X_glob_val.to(device)
    y_val = y_val.to(device)

    outputs = model(X_seq_val, X_glob_val)

    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)
    correct = (preds.cpu().numpy() == y_val.cpu().numpy()).sum()
    total=len(preds.cpu().numpy())
print("Predictions:", preds.cpu().numpy())
print("True labels:", y_val.cpu().numpy())
print(f'correct {correct} out of {total}' )

Predictions: [1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 2 2 1
 2 1 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 1 2 2 1 2 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 1 1 2 2 2 2 1 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 2 2 2 2
 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1]
True labels: [0 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0
 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 2 2 2 2 2
 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1
 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1
 1 0 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0
 0 0 0 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 2 2 2 2
 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1]
correc